# Task 1

**Task 1: Build a Relational Schema
Model an online movie rental service. Design a normalized relational schema using pandas DataFrames for the following entities:
Step 1: Create these tables:
customers with columns: customer_id, name, email, city, signup_date
movies with columns: movie_id, title, genre, release_year, rating (1-5 scale)
rentals with columns: rental_id, customer_id, movie_id, rental_date, return_date, price
Populate with at least 8 customers, 10 movies, and 25 rentals. Make sure some customers have multiple rentals and some movies are rented multiple times.**

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [12]:
customers_data = {
    'customer_id': range(1, 9),
    'name': ['Ali', 'Leyla', 'Murad', 'Aysel', 'Fuad', 'Gunel', 'Zaur', 'Fidan'],
    'email': ['ali@mail.com', 'leyla@mail.com', 'murad@mail.com', 'aysel@mail.com', 
              'fuad@mail.com', 'gunel@mail.com', 'zaur@mail.com', 'fidan@mail.com'],
    'city': ['Baku', 'Ganja', 'Baku', 'Sumqayit', 'Baku', 'Ganja', 'Lankaran', 'Baku'],
    'signup_date': pd.to_datetime(['2025-01-10', '2025-01-15', '2025-02-01', '2025-02-05', 
                                   '2025-02-10', '2025-02-15', '2025-02-20', '2025-03-01'])
}
customers = pd.DataFrame(customers_data)#.set_index('customer_id')
customers

,customer_id,name,email,city,signup_date
0,1,Ali,ali@mail.com,Baku,2025-01-10
1,2,Leyla,leyla@mail.com,Ganja,2025-01-15
2,3,Murad,murad@mail.com,Baku,2025-02-01
3,4,Aysel,aysel@mail.com,Sumqayit,2025-02-05
4,5,Fuad,fuad@mail.com,Baku,2025-02-10
5,6,Gunel,gunel@mail.com,Ganja,2025-02-15
6,7,Zaur,zaur@mail.com,Lankaran,2025-02-20
7,8,Fidan,fidan@mail.com,Baku,2025-03-01


In [13]:
movies_data = {
    'movie_id': range(1, 11),
    'title': ['Inception', 'The Matrix', 'Interstellar', 'The Godfather', 'Pulp Fiction', 
              'The Dark Knight', 'Gladiator', 'Joker', 'Parasite', 'The Whale'],
    'genre': ['Sci-Fi', 'Sci-Fi', 'Sci-Fi', 'Crime', 'Crime', 'Action', 'Action', 'Drama', 'Thriller', 'Drama'],
    'release_year': [2010, 1999, 2014, 1972, 1994, 2008, 2000, 2019, 2019, 2022],
    'rating': [5, 5, 4, 5, 4, 5, 4, 4, 5, 3]
}
movies = pd.DataFrame(movies_data)#.set_index('movie_id')
movies

,movie_id,title,genre,release_year,rating
0,1,Inception,Sci-Fi,2010,5
1,2,The Matrix,Sci-Fi,1999,5
2,3,Interstellar,Sci-Fi,2014,4
3,4,The Godfather,Crime,1972,5
4,5,Pulp Fiction,Crime,1994,4
5,6,The Dark Knight,Action,2008,5
6,7,Gladiator,Action,2000,4
7,8,Joker,Drama,2019,4
8,9,Parasite,Thriller,2019,5
9,10,The Whale,Drama,2022,3


In [32]:
np.random.seed(42)
rentals_data = {
    'rental_id': range(1, 26),
    'customer_id': np.random.choice(customers['customer_id'], 25), 
    'movie_id': np.random.choice(movies['movie_id'], 25),     
    'rental_date': [datetime(2026, 2, 1) + timedelta(days=i) for i in range(25)],
    'price': np.random.uniform(2.5, 7.5, 25).round(2)
}
rentals = pd.DataFrame(rentals_data)
rentals.head()

,rental_id,customer_id,movie_id,rental_date,price
0,1,7,6,2026-02-01,4.02
1,2,4,2,2026-02-02,2.99
2,3,5,5,2026-02-03,5.92
3,4,7,1,2026-02-04,4.70
4,5,3,10,2026-02-05,3.11


In [33]:
rentals['return_date']= rentals['rental_date'] + timedelta(days=3)

**Step 2: Write and execute these queries using pandas operations:
Find all movies rented by a specific customer (requires joining rentals with movies).
Find the top 5 most-rented movies (group and count).
Compute the total revenue per genre (join rentals with movies, group by genre, sum price).
Find customers who have never rented a movie rated below 3 (multi-step filtering).**

In [34]:
target_customer_id = 7
customer_movies = pd.merge(rentals, movies, on='movie_id')
customer_movies[customer_movies['customer_id']==target_customer_id]

,rental_id,customer_id,movie_id,rental_date,price,return_date,title,genre,release_year,rating
0,1,7,6,2026-02-01,4.02,2026-02-04,The Dark Knight,Action,2008,5
3,4,7,1,2026-02-04,4.70,2026-02-07,Inception,Sci-Fi,2010,5
8,9,7,10,2026-02-09,3.79,2026-02-12,The Whale,Drama,2022,3
11,12,7,4,2026-02-12,5.10,2026-02-15,The Godfather,Crime,1972,5


In [35]:
merged_df = pd.merge(rentals, movies, on='movie_id')
top_movies = merged_df.groupby('title').size().reset_index(name='rental_count')
top_5 = top_movies.sort_values(by='rental_count', ascending=False).head(5)

print("top5:")
print(top_5)

top5:
          title  rental_count
3      Parasite             4
0     Gladiator             3
2  Interstellar             3
4  Pulp Fiction             3
8     The Whale             3


In [36]:
customer_movies.groupby("genre")["price"].sum()

genre
Action      27.37
Crime       28.82
Drama       11.34
Sci-Fi      37.42
Thriller    16.12
Name: price, dtype: float64

In [37]:
merged_df = pd.merge(rentals, movies, on='movie_id')

bad_rentals = merged_df[merged_df['rating'] < 3]

customers_with_bad_ratings = bad_rentals['customer_id'].unique()
customers_with_bad_ratings

clean_customers = customers[~customers['customer_id'].isin(customers_with_bad_ratings)]

print("who have never rented a movie rated below 3")
print(clean_customers[['name', 'email', 'city']])

who have never rented a movie rated below 3
    name           email      city
0    Ali    ali@mail.com      Baku
1  Leyla  leyla@mail.com     Ganja
2  Murad  murad@mail.com      Baku
3  Aysel  aysel@mail.com  Sumqayit
4   Fuad   fuad@mail.com      Baku
5  Gunel  gunel@mail.com     Ganja
6   Zaur   zaur@mail.com  Lankaran
7  Fidan  fidan@mail.com      Baku


**Step 3: Demonstrate normalization. Show what the data would look like as a single denormalized table (join all three tables). Write a markdown cell explaining what redundancy appears and why the normalized version is preferable for updates.**

In [40]:
# Joining all three tables into one (JOIN)
denormalized_df = pd.merge(rentals, customers, on='customer_id')
denormalized_df = pd.merge(denormalized_df, movies, on='movie_id')

print("Denormalized Unified Table:")
print(denormalized_df[['rental_id', 'name', 'city', 'title', 'genre', 'price']].head(10))

Denormalized Unified Table:
   rental_id   name      city            title     genre  price
0          1   Zaur  Lankaran  The Dark Knight    Action   4.02
1          2  Aysel  Sumqayit       The Matrix    Sci-Fi   2.99
2          3   Fuad      Baku     Pulp Fiction     Crime   5.92
3          4   Zaur  Lankaran        Inception    Sci-Fi   4.70
4          5  Murad      Baku        The Whale     Drama   3.11
5          6  Fidan      Baku  The Dark Knight    Action   4.98
6          7   Fuad      Baku         Parasite  Thriller   2.67
7          8   Fuad      Baku        Inception    Sci-Fi   7.05
8          9   Zaur  Lankaran        The Whale     Drama   3.79
9         10  Leyla     Ganja     Interstellar    Sci-Fi   5.81


*While a denormalized table can be faster for certain analytical tasks (OLAP) because it avoids complex joins, a normalized relational schema is the "gold standard" for transactional systems (OLTP) where data is frequently updated and accuracy is critical.*

# Task 2

**Step 1: Create a list of customer documents (Python dictionaries) where each document contains the customer's information and a nested list of their rental history, with movie details embedded in each rental entry.**

In [43]:
import json
customer_docs_full = []

for _, cust in customers.iterrows():

    doc = {
        "customer_id": int(cust['customer_id']),
        "name": cust['name'],
        "email": cust['email'],
        "city": cust['city'],
        "signup_date": cust['signup_date'].strftime('%Y-%m-%d'),
        "rentals": []
    }

    cust_rentals = rentals[rentals['customer_id'] == cust['customer_id']]
    
    for _, rent in cust_rentals.iterrows():
        
        movie_info = movies[movies['movie_id'] == rent['movie_id']].iloc[0]

        rental_entry = {
            "rental_id": int(rent['rental_id']),
            "rental_date": rent['rental_date'].strftime('%Y-%m-%d'),
            "price": float(rent['price']),
            "movie": {
                "title": movie_info['title'],
                "genre": movie_info['genre'],
                "rating": int(movie_info['rating'])
            }
        }
        doc["rentals"].append(rental_entry)
    
    customer_docs_full.append(doc)

print(json.dumps(customer_docs_full[1], indent=4))

{
    "customer_id": 2,
    "name": "Leyla",
    "email": "leyla@mail.com",
    "city": "Ganja",
    "signup_date": "2025-01-15",
    "rentals": [
        {
            "rental_id": 10,
            "rental_date": "2026-02-10",
            "price": 5.81,
            "movie": {
                "title": "Interstellar",
                "genre": "Sci-Fi",
                "rating": 4
            }
        },
        {
            "rental_id": 23,
            "rental_date": "2026-02-23",
            "price": 2.73,
            "movie": {
                "title": "Parasite",
                "genre": "Thriller",
                "rating": 5
            }
        }
    ]
}


In [46]:
target_id = 2
# Locate the customer and access their nested rentals list
customer = next(c for c in customer_docs_full if c['customer_id'] == target_id)

print(f"Movies rented by {customer['name']}:")
for r in customer['rentals']:
    print(f"- {r['movie']['title']} (Genre: {r['movie']['genre']})")

Movies rented by Leyla:
- Interstellar (Genre: Sci-Fi)
- Parasite (Genre: Thriller)


In [47]:
from collections import Counter

movie_counts = Counter()
for doc in customer_docs_full:
    for r in doc['rentals']:
        movie_counts[r['movie']['title']] += 1

print("Top 5 Most-Rented Movies:")
for title, count in movie_counts.most_common(5):
    print(f"{title}: {count} rentals")

Top 5 Most-Rented Movies:
Parasite: 4 rentals
Interstellar: 3 rentals
The Whale: 3 rentals
Gladiator: 3 rentals
The Matrix: 3 rentals


In [48]:
genre_revenue = {}
for doc in customer_docs_full:
    for r in doc['rentals']:
        genre = r['movie']['genre']
        price = r['price']
        genre_revenue[genre] = genre_revenue.get(genre, 0) + price

print("Total Revenue per Genre:")
for genre, rev in genre_revenue.items():
    print(f"{genre}: ${rev:.2f}")

Total Revenue per Genre:
Sci-Fi: $37.42
Thriller: $16.12
Drama: $11.34
Action: $27.37
Crime: $28.82


In [49]:
clean_customers_list = []

for doc in customer_docs_full:
    # Check if any movie in their rental history has a rating less than 3
    low_rated_rentals = [r for r in doc['rentals'] if r['movie']['rating'] < 3]
    
    # If the list is empty, they are a "clean" customer
    if not low_rated_rentals:
        clean_customers_list.append(doc['name'])

print("Customers who only rent high-rated movies (Rating >= 3):")
print(clean_customers_list)

Customers who only rent high-rated movies (Rating >= 3):
['Ali', 'Leyla', 'Murad', 'Aysel', 'Fuad', 'Gunel', 'Zaur', 'Fidan']


**Step 3: Write a markdown cell comparing the document queries to the relational queries. Which queries were easier to write? Which were harder? For which query pattern does the document model have a clear advantage?**

* 1. Which queries were easier to write? - relational
2.Which queries were harder to write? - document
3.The document model has a clear advantage for One-to-Many relationships where you always need the "child" data (rentals) whenever you access the "parent" (customer). *

# Task 3

**Step 1: Create a graph using Python dictionaries where:
Nodes represent customers, movies, and genres.
Edges represent relationships: rented (customer → movie), belongs_to (movie → genre), lives_in (customer → city).**

In [ ]:
# Nodes: Representing entities (customers, movies, genres, and cities)
nodes = {
    # Customers
    "c001": {"type": "customer", "name": "Alice"},
    "c002": {"type": "customer", "name": "Bob"},
    "c003": {"type": "customer", "name": "Charlie"},

    # Movies
    "m001": {"type": "movie", "title": "Inception"},
    "m002": {"type": "movie", "title": "The Dark Knight"},
    "m003": {"type": "movie", "title": "Interstellar"},

    # Genres
    "action": {"type": "genre", "name": "Action"},
    "scifi": {"type": "genre", "name": "Sci-Fi"},

    # Cities
    "london": {"type": "city", "name": "London"},
    "new_york": {"type": "city", "name": "New York"}
}

# Edges: Representing relationships
# Format: (source_id, relationship_type, target_id)
edges = [
    # Rental relationships (customer -> movie)
    ("c001", "rented", "m001"),
    ("c001", "rented", "m003"),
    ("c002", "rented", "m002"),
    ("c003", "rented", "m001"),

    # Genre relationships (movie -> genre)
    ("m001", "belongs_to", "scifi"),
    ("m002", "belongs_to", "action"),
    ("m003", "belongs_to", "scifi"),

    # Location relationships (customer -> city)
    ("c001", "lives_in", "london"),
    ("c002", "lives_in", "new_york"),
    ("c003", "lives_in", "london")
]

**Step 2: Write traversal functions to answer:
"Which movies did customer X rent?" — one hop from customer through rented edges.
"Which genres does customer X prefer?" — two hops: customer → movies → genres, then count.
"Which customers share a genre preference with customer X?" — three hops: customer X → movies → genres → other customers who rented movies in the same genre.
"Recommend movies for customer X" — find movies in their preferred genres that they have not yet rented.**

In [54]:
from collections import Counter

# 1. Which movies did customer X rent? (One hop)
def get_rented_movies(customer_id):
    # Filter edges where source is customer and relation is 'rented'
    movie_ids = [target for source, rel, target in edges 
                 if source == customer_id and rel == "rented"]
    return [nodes[m_id]["title"] for m_id in movie_ids]

# 2. Which genres does customer X prefer? (Two hops: customer -> movies -> genres)
def get_preferred_genres(customer_id):
    genres = []
    # Hop 1: Find rented movies
    movie_ids = [target for source, rel, target in edges 
                 if source == customer_id and rel == "rented"]
    
    # Hop 2: Find genres for those movies
    for m_id in movie_ids:
        movie_genres = [target for source, rel, target in edges 
                        if source == m_id and rel == "belongs_to"]
        genres.extend(movie_genres)
    
    # Return count of genres (most frequent first)
    return Counter(genres)

# 3. Which customers share a genre preference with customer X? (Three hops)
def get_similar_customers(customer_id):
    my_genres = set(get_preferred_genres(customer_id).keys())
    similar_customers = set()

    for other_id, info in nodes.items():
        if info["type"] == "customer" and other_id != customer_id:
            other_genres = set(get_preferred_genres(other_id).keys())
            # Check if there is an intersection between genres
            if my_genres & other_genres:
                similar_customers.add(info["name"])
    
    return list(similar_customers)

# 4. Recommend movies for customer X
def recommend_movies(customer_id):
    preferred_genres = set(get_preferred_genres(customer_id).keys())
    already_rented = set([target for source, rel, target in edges 
                          if source == customer_id and rel == "rented"])
    
    recommendations = []
    for m_id, info in nodes.items():
        if info["type"] == "movie" and m_id not in already_rented:
            # Check if movie belongs to a preferred genre
            movie_genres = set([target for source, rel, target in edges 
                                if source == m_id and rel == "belongs_to"])
            if preferred_genres & movie_genres:
                recommendations.append(info["title"])
                
    return recommendations

**Step 3: Write a markdown cell explaining why the recommendation query (Task 3, question 4) would be difficult to express in SQL or with documents. What makes the graph model natural for this kind of traversal?**

* SQL (Relational) Complexity: Recommendations require connecting several tables (Customers $\to$ Rentals $\to$ Movies $\to$ Genres). In SQL, this involves multiple expensive JOIN operations. As the chain of "hops" grows, performance drops and the query becomes a nightmare of nested logic.Document (NoSQL) Rigidity: Documents like JSON are great for single profiles but poor for connections. To find "users with similar tastes," you’d have to fetch and scan thousands of separate documents, which is highly inefficient for real-time links.The Graph Advantage: In a graph, relationships are "first-class citizens." The database doesn't search for a connection; it simply follows a physical pointer (Index-Free Adjacency). This makes "walking" through data—from a person to a genre to a new movie—instant and natural. *

# Task 4

**Step 1: Create a DataFrame representing a log of 5,000 e-commerce transactions with columns: transaction_id, customer_id, product_id, amount, payment_method, timestamp, status (completed/refunded/pending).**

In [56]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set seed for reproducibility
np.random.seed(42)

# Configuration
num_records = 5000

# 1. Generate IDs
transaction_ids = np.arange(10000, 10000 + num_records)
customer_ids = np.random.randint(500, 2000, size=num_records)  # Reoccurring customers
product_ids = np.random.randint(1, 500, size=num_records)

# 2. Generate Amounts (Log-normal distribution for realistic pricing)
amounts = np.round(np.random.lognormal(mean=3.5, sigma=0.8, size=num_records), 2)

# 3. Categorical Data with weighted probabilities
payment_methods = np.random.choice(
    ['Credit Card', 'PayPal', 'Debit Card', 'Apple Pay'], 
    size=num_records
)

status_options = ['completed', 'refunded', 'pending']
status_weights = [0.85, 0.05, 0.10]  # 85% completed, 5% refunded, 10% pending
statuses = np.random.choice(status_options, size=num_records, p=status_weights)

# 4. Generate Timestamps (Over the last 30 days)
start_date = datetime(2026, 2, 1)
timestamps = [start_date + timedelta(
    days=np.random.randint(0, 30),
    hours=np.random.randint(0, 24),
    minutes=np.random.randint(0, 60)
) for _ in range(num_records)]

# 5. Assemble the DataFrame
df = pd.DataFrame({
    'transaction_id': transaction_ids,
    'customer_id': customer_ids,
    'product_id': product_ids,
    'amount': amounts,
    'payment_method': payment_methods,
    'timestamp': timestamps,
    'status': statuses
})

# Display the first few rows
df.head()

,transaction_id,customer_id,product_id,amount,payment_method,timestamp,status
0,10000,1626,343,12.79,Credit Card,2026-02-03 02:17:00,completed
1,10001,1959,408,31.82,Apple Pay,2026-02-13 00:36:00,completed
2,10002,1360,139,14.34,Credit Card,2026-02-28 03:15:00,completed
3,10003,1794,110,48.36,Debit Card,2026-02-15 17:48:00,completed
4,10004,1630,482,41.79,Debit Card,2026-02-14 13:18:00,completed


**Step 2: Simulate OLTP-style operations:
Look up a single transaction by its ID (simulate a point query).
Insert a new transaction (append a row).
Update the status of a transaction from "pending" to "completed."
Check that the transaction is valid: the amount must be positive and the status must be one of the allowed values (simulate a consistency check).**

In [57]:
target_id = 10008
df[df['transaction_id']==target_id]

,transaction_id,customer_id,product_id,amount,payment_method,timestamp,status
8,10008,966,191,11.11,Debit Card,2026-02-19 16:40:00,completed


In [58]:
df.loc[len(df)] = [15001, 999,42,125.50,'Credit Card',datetime.now(),'pending']

In [59]:
df.loc[len(df)-1]

transaction_id                         15001
customer_id                              999
product_id                                42
amount                                 125.5
payment_method                   Credit Card
timestamp         2026-03-04 11:50:05.038719
status                               pending
Name: 5000, dtype: object

In [61]:
df.loc[df['transaction_id'] == 15001, 'status'] = 'completed'

In [63]:
def validate_transaction(row):
    valid_statuses = {'completed', 'refunded', 'pending'}
    
    # Check  Positive Amount
    is_amount_valid = row['amount'] > 0
    
    # chcek Valid Status
    is_status_valid = row['status'] in valid_statuses
    
    return is_amount_valid and is_status_valid

# Example validation on our new record
is_valid = validate_transaction(df[df['transaction_id'] == 15001].iloc[0])
print(f"Transaction 15001 is valid: {is_valid}")

Transaction 15001 is valid: True


**Step 2: Simulate OLTP-style operations:
Look up a single transaction by its ID (simulate a point query).
Insert a new transaction (append a row).
Update the status of a transaction from "pending" to "completed."
Check that the transaction is valid: the amount must be positive and the status must be one of the allowed values (simulate a consistency check).**

In [65]:
total_revune = df.groupby('payment_method')['amount'].sum()
total_revune

payment_method
Apple Pay      56448.28
Credit Card    54543.57
Debit Card     58143.09
PayPal         59641.36
Name: amount, dtype: float64

In [74]:
df['month'] = df['timestamp'].dt.to_period('M')
avg_by_month = df.groupby('month')['amount'].mean()

print(avg_by_month)

month
2026-02    45.641135
2026-03    47.141086
Freq: M, Name: amount, dtype: float64


In [66]:
df.groupby('customer_id')['amount'].sum().head(10)

customer_id
500    237.94
501    188.88
502    232.53
503     29.48
504     86.89
505    102.48
506    280.10
507    139.26
508     89.60
509    164.68
Name: amount, dtype: float64

In [72]:
df_groupedby_status = df.groupby('status')['status'].count()

refunded_count = df_groupedby_status.get('refunded', 0)

total_transactions = df_groupedby_status.sum()

refund_rate = (refunded_count / total_transactions) * 100

print(f"Refund Rate: {refund_rate:.2f}%")

Refund Rate: 5.12%


**OLTP uses row-major storage because its workload involves handling individual records (Inserts/Updates). By storing all attributes of a single transaction together, the system can read or write an entire row in a single, fast I/O operation. Conversely, OLAP uses column-major storage for high-speed aggregations. Since analytical queries usually only need specific columns (e.g., "Average Amount"), storing columns together allows the system to skip irrelevant data, reducing memory usage and enabling better compression.**

# Task 5

**For each of the following real-world scenarios, recommend a data model (relational, document, or graph) and a processing paradigm (OLTP or OLAP). Write 3-4 sentences justifying each choice.
A hospital patient record system where doctors need to quickly look up a patient's complete medical history.
A social network that needs to suggest "friends of friends" connections.
A financial reporting system that computes monthly revenue breakdowns across product categories and regions.
An IoT platform that receives sensor readings from 10,000 devices and needs to detect anomalies in real time.
A content management system where each article has a different set of metadata fields.**

* In a hospital patient record system, the Document Model combined with OLTP is the best fit. This allows a patient's entire medical history—vitals, lab results, and prescriptions—to be stored in a single, nested record for instant retrieval by doctors during a consultation. Because medical staff need to update these records in real-time, the system must prioritize high-speed, row-level transactions. *

* For a social network suggesting "friends of friends," a Graph Model with OLTP is the superior choice. Social connections are naturally nodes and edges; a graph database can traverse these relationships much faster than traditional tables. This is an OLTP workload because these suggestions need to be generated and updated instantly as users browse their profiles. *

* A financial reporting system requires a Relational Model and OLAP processing. Instead of individual updates, this system scans millions of historical rows to compute revenue trends across regions or product categories. The structured nature of relational tables ensures the mathematical precision needed for audits, while the OLAP paradigm optimizes the heavy aggregation queries. * 

* An IoT platform handling 10,000 devices works best with a Time-Series Relational Model and OLTP. The data is a high-velocity stream of timestamps and values that must be indexed sequentially. An OLTP-style engine is required to process these incoming writes and run immediate checks to catch anomalies, like a sudden temperature spike, the moment they occur.*

* Finally, a Content Management System (CMS) should use a Document Model and OLTP. Since different articles (like a recipe vs. a tech review) have different metadata fields, a schema-less document model provides the necessary flexibility. It functions as an OLTP system because the primary actions—writing a draft or loading a specific page—are individual, high-frequency operations. *